# Baseline Models for Rent Price Prediction

This notebook demonstrates the creation of baseline models for rent price prediction.

**Key steps:**
- Data loading and feature selection
- Log-transforming the target variable (`price`) for better model performance
- Using cross-validation to evaluate models
- Comparing baseline models using RMSLE metric (on original price scale), because RMSLE penalized the underestimate more than overestimate

## Imports

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import root_mean_squared_log_error, make_scorer
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

## Load Data and Select Features

In [2]:
data = pd.read_csv('../data/processed/train_df.csv')
data.sample(3)

,id,price,address,coordinates,region,subway,rooms,footage,floor,features,residential,neighborhood,description,detail,attributes,full_area,living_area,kitchen_area,num_storeys,district,lon,lat,neighborhood_1,neighborhood_2,neighborhood_3,neighborhood_4,neighborhood_5,price_bin
761,11417176,790,"Довженка Олександра вул., 16","30.44789505,50.46186447","Київ,Шевченківський р-н",Шулявська,2,54 / 25 / 12 м²,10,"['Бетонно монолітний', 'Роздільне', 'Дизайнерс...",NaN,[],Актуально! Можливі перегляди!\n\nПропонуємо в ...,"Будинок - Бетонно монолітний, в квартирі 2 кім...","['Посудомийна машина', 'Кондиціонер', 'Мікрохв...",54.0,25.0,12.0,16,Шевченківський,30.447895,50.461864,NaN,NaN,NaN,NaN,NaN,upper_mid
1858,11314406,2600,"Ярославів Вал вул., 15а","30.50830269,50.45044327","Київ,Шевченківський р-н",Золоті Ворота,3,130 / 80 / 30 м²,8,"['Стара панель', 'Роздільне', 'Дизайнерський р...",NaN,"['Афанасівський яр', 'Старий Київ', 'Ботанічни...",Аренда 3к квартиры (130м) по ул. Ярославов Вал...,"Будинок - Стара панель, в квартирі 3 кімнати. ...","['Камін', 'Посудомийна машина', 'Душова кабіна...",130.0,80.0,30.0,10,Шевченківський,30.508303,50.450443,Афанасівський яр,Старий Київ,Ботанічний сад ім. акад. О. В. Фоміна,NaN,NaN,ultra_luxury
6392,11361118,270,"Закревського Миколи вул., 87г","30.62594414,50.52608871","Київ,Деснянський р-н",NaN,3,82 / 54 / 8 м²,12,"['Українська панель', 'Роздільне', 'Хороший ст...",NaN,['Троєщина'],"вул. Закревського 87г, поруч дитячий садочок,...","Будинок - Українська панель, в квартирі 3 кімн...",NaN,82.0,54.0,8.0,16,Деснянський,30.625944,50.526089,Троєщина,NaN,NaN,NaN,NaN,budget


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9140 entries, 0 to 9139
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              9140 non-null   int64  
 1   price           9140 non-null   int64  
 2   address         9140 non-null   object 
 3   coordinates     9032 non-null   object 
 4   region          9140 non-null   object 
 5   subway          7375 non-null   object 
 6   rooms           9140 non-null   int64  
 7   footage         9140 non-null   object 
 8   floor           9140 non-null   int64  
 9   features        9140 non-null   object 
 10  residential     4367 non-null   object 
 11  neighborhood    9140 non-null   object 
 12  description     9140 non-null   object 
 13  detail          9140 non-null   object 
 14  attributes      5501 non-null   object 
 15  full_area       9140 non-null   float64
 16  living_area     9140 non-null   float64
 17  kitchen_area    9140 non-null   f

In [4]:
black_list = ['id', 'price', 'price_bin']

numerical_feats = [col for col in data.select_dtypes(exclude='object').columns
                   if col not in black_list]

In [5]:
data[numerical_feats].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9140 entries, 0 to 9139
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rooms         9140 non-null   int64  
 1   floor         9140 non-null   int64  
 2   full_area     9140 non-null   float64
 3   living_area   9140 non-null   float64
 4   kitchen_area  9140 non-null   float64
 5   num_storeys   9140 non-null   int64  
 6   lon           9032 non-null   float64
 7   lat           9032 non-null   float64
dtypes: float64(5), int64(3)
memory usage: 571.4 KB


In [6]:
X = data[numerical_feats]
y = np.log1p(data['price'])

## Custom RMSLE Scorer for Log-Transformed Target

In [ ]:
def rmsle_exp(y_true, y_pred):
    y_true_exp = np.expm1(y_true)
    y_pred_exp = np.expm1(y_pred)
    return root_mean_squared_log_error(y_true, y_pred)

rmsle_scorer = make_scorer(rmsle_exp, greater_is_better=False)

##  Cross-Validation Helper

In [8]:
def cv_results(model, X, y, cv=5):
    results = cross_validate(model, X, y,
                             scoring=rmsle_scorer,
                             cv=cv, return_train_score=True)
    df_results = (pd.DataFrame(results)[['train_score', 'test_score']]
                  .apply(lambda x: -x))
    scores = results["test_score"]
    print(
        "Mean RMSLE (original scale) on cross-validation: "
        f"{-scores.mean():.4f} ± {scores.std():.4f}"
    )
    return df_results, -scores.mean(), scores.std()

## Baseline Models

In [9]:
results = []

### DummyRegressor (mean)

In [10]:
model = DummyRegressor(strategy='mean')
df_result, mean_score, mean_std = cv_results(model, X, y)
results.append(('DummyRegressor-mean', mean_score, mean_std))
df_result.round(4)

Mean RMSLE (original scale) on cross-validation: 0.0967 ± 0.0020


,train_score,test_score
0,0.0968,0.0968
1,0.0960,0.0996
2,0.0970,0.0958
3,0.0975,0.0937
4,0.0964,0.0978


### DummyRegressor (median)

In [11]:
model = DummyRegressor(strategy='median')
_, median_score, median_std = cv_results(model, X, y)
results.append(('DummyRegressor-median', median_score, median_std))

Mean RMSLE (original scale) on cross-validation: 0.0968 ± 0.0021


### Linear Regression

In [12]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

_, lr_score, lr_std = cv_results(pipeline, X, y)
results.append(('LinearRegression', lr_score, lr_std))

Mean RMSLE (original scale) on cross-validation: 0.0635 ± 0.0010


### ExtraTreesRegressor

In [13]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('model', ExtraTreesRegressor(max_depth=6, random_state=7))
])

_, et_score, et_std = cv_results(pipeline, X, y)
results.append(('ExtraTreesRegressor', et_score, et_std))

Mean RMSLE (original scale) on cross-validation: 0.0539 ± 0.0010


### RandomForestRegressor

In [14]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('model', RandomForestRegressor(max_depth=6, random_state=7))
])

_, rf_score, rf_std = cv_results(pipeline, X, y)
results.append(('RandomForestRegressor', rf_score, rf_std))

Mean RMSLE (original scale) on cross-validation: 0.0477 ± 0.0008


### XGBoost

In [15]:
parameters_xgb = {
    'objective': 'reg:squarederror',
    'eta': 0.1,
    'verbosity': 1,
    'seed': 7,
    'tree_method': 'hist',
    'grow_policy': 'lossguide',
    'max_depth': 6,
    'subsample': 0.7,
}
model = xgb.XGBRegressor(**parameters_xgb)

_, xgb_score, xgb_std = cv_results(model, X, y)
results.append(('XGBRegressor', xgb_score, xgb_std))

Mean RMSLE (original scale) on cross-validation: 0.0376 ± 0.0010


### LightGBM

In [16]:
parameters_lgb = {
    'objective': 'regression',
    'learning_rate': 0.1,
    'random_state': 7,
    'verbose': 0,
    'force_col_wise': True,
    'max_depth': 6,
    'num_leaves': 7,
    'subsample': 0.7,
    'subsample_freq': 1
}

model = lgb.LGBMRegressor(**parameters_lgb)

_, lgb_score, lgb_std = cv_results(model, X, y)
results.append(('LGBMRegressor', lgb_score, lgb_std))

Mean RMSLE (original scale) on cross-validation: 0.0422 ± 0.0010


## Summary Table

In [20]:
results_df = pd.DataFrame(results, columns=['Model', 'Mean RMSLE', 'Std'])
results_df = results_df.sort_values('Mean RMSLE')
results_df.reset_index(drop=True, inplace=True)
results_df.style.highlight_min(subset=['Mean RMSLE'], color='tan')

,Model,Mean RMSLE,Std
0,XGBRegressor,0.037594,0.000977
1,LGBMRegressor,0.042222,0.001039
2,RandomForestRegressor,0.047669,0.000757
3,ExtraTreesRegressor,0.053888,0.001024
4,LinearRegression,0.063541,0.000974
5,DummyRegressor-mean,0.096739,0.001970
6,DummyRegressor-median,0.096780,0.002113


- We established several baseline models for rent price prediction using only numerical features and log-transformed target values.
- Tree-based models (RandomForest, ExtraTrees, XGBoost, LightGBM) significantly outperform simple baselines (Dummy, Linear Regression) in terms of RMSLE.
- The best baseline RMSLE is achieved by XGBoost (0.0376), followed closely by LightGBM (0.0422) and RandomForest (0.0477).

**Conclusions:**
- Keep DummyRegressors as baseline references for future experiments.
- Perform feature engineering, incorporate categorical, text, and geospatial features.
- Proceed with hyperparameter tuning for XGB and LGBM as the leading baseline to further improve performance.